# Практическая работа №4. Задания — Вариант 7

Выполняются все пункты варианта 7 по последнему номеру зачетной книжки:

1. Для каждого пользователя посчитать средний интервал между событиями (`delta_since_prev`).
2. Найти пользователей с минимальным и максимальным средним интервалом.
3. Интерпретировать, что это может означать с точки зрения активности.

## Подготовка

Загрузка событий, перевод времени в московскую зону и расчёт интервала между соседними событиями пользователя.

In [1]:
import pandas as pd
import numpy as np

df_events_records = pd.read_json("../shared/data/events_records.json", orient="records")
df_events_lines = pd.read_json("../shared/data/events_lines.json", lines=True)

df_events = pd.concat(
    [df_events_records, df_events_lines],
    ignore_index=True
)

df_events["event_time"] = pd.to_datetime(df_events["event_time"])
df_events["event_time_msk"] = (
    df_events["event_time"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("Europe/Moscow")
)

df_events_user = df_events.sort_values(["user_id", "event_time_msk"]).copy()

df_events_user["prev_event_time"] = (
    df_events_user
    .groupby("user_id")["event_time_msk"]
    .shift(1)
)
df_events_user["delta_since_prev"] = (
    df_events_user["event_time_msk"] - df_events_user["prev_event_time"]
)

print(df_events_user[["user_id", "event_time_msk", "prev_event_time", "delta_since_prev"]].head(10))

      user_id            event_time_msk           prev_event_time  \
930      1001 2024-01-03 08:57:47+03:00                       NaT   
1591     1001 2024-01-04 10:42:32+03:00 2024-01-03 08:57:47+03:00   
599      1001 2024-01-06 05:06:36+03:00 2024-01-04 10:42:32+03:00   
1215     1001 2024-01-07 03:43:21+03:00 2024-01-06 05:06:36+03:00   
676      1001 2024-01-07 10:17:51+03:00 2024-01-07 03:43:21+03:00   
893      1001 2024-01-12 21:36:41+03:00 2024-01-07 10:17:51+03:00   
1411     1001 2024-01-24 05:04:00+03:00 2024-01-12 21:36:41+03:00   
1644     1001 2024-01-25 03:41:23+03:00 2024-01-24 05:04:00+03:00   
1152     1001 2024-01-26 13:25:08+03:00 2024-01-25 03:41:23+03:00   
1413     1001 2024-01-27 21:28:01+03:00 2024-01-26 13:25:08+03:00   

     delta_since_prev  
930               NaT  
1591  1 days 01:44:45  
599   1 days 18:24:04  
1215  0 days 22:36:45  
676   0 days 06:34:30  
893   5 days 11:18:50  
1411 11 days 07:27:19  
1644  0 days 22:37:23  
1152  1 days 09:43:45  


### Задание 1. Для каждого пользователя посчитать средний интервал между событиями (`delta_since_prev`)

In [2]:
# Первое событие пользователя не имеет предыдущего (NaT) — mean() его пропускает
user_intervals = (
    df_events_user
    .groupby("user_id")
    .agg(
        events_count=("event_id", "count"),
        mean_interval=("delta_since_prev", "mean"),
        median_interval=("delta_since_prev", "median"),
    )
)
user_intervals["mean_interval_hours"] = user_intervals["mean_interval"] / pd.Timedelta(hours=1)

print("Пользователей:", user_intervals.shape[0])
print(user_intervals.sort_values("mean_interval").head(10))
print(user_intervals["mean_interval_hours"].describe())

Пользователей: 100
         events_count          mean_interval        median_interval  \
user_id                                                               
1040               60 1 days 09:24:31.271186        1 days 03:12:36   
1091               51 1 days 14:05:49.100000        1 days 07:56:04   
1072               56 1 days 14:08:07.254545        1 days 06:44:18   
1075               52 1 days 15:42:39.019607        0 days 23:44:40   
1076               53 1 days 16:33:15.365384 0 days 23:16:17.500000   
1099               51 1 days 17:16:50.760000 1 days 01:29:46.500000   
1026               44 1 days 18:09:42.418604        1 days 09:32:43   
1060               50 1 days 18:19:36.346938        1 days 01:03:25   
1069               47 1 days 19:27:12.347826        0 days 22:10:08   
1085               48 1 days 19:36:10.319148        1 days 02:25:30   

         mean_interval_hours  
user_id                       
1040               33.408686  
1091               38.096972  
1072

### Задание 2. Найти пользователей с минимальным и максимальным средним интервалом

In [3]:
min_user = user_intervals["mean_interval"].idxmin()
max_user = user_intervals["mean_interval"].idxmax()

print("Минимальный средний интервал:")
print(user_intervals.loc[[min_user]])
print()
print("Максимальный средний интервал:")
print(user_intervals.loc[[max_user]])

# Для сравнения — по 5 самых «частых» и самых «редких» пользователей
print()
print(user_intervals.nsmallest(5, "mean_interval_hours")[["events_count", "mean_interval"]])
print(user_intervals.nlargest(5, "mean_interval_hours")[["events_count", "mean_interval"]])

Минимальный средний интервал:
         events_count          mean_interval median_interval  \
user_id                                                        
1040               60 1 days 09:24:31.271186 1 days 03:12:36   

         mean_interval_hours  
user_id                       
1040               33.408686  

Максимальный средний интервал:
         events_count          mean_interval median_interval  \
user_id                                                        
1082               27 3 days 11:01:53.423076 2 days 19:00:32   

         mean_interval_hours  
user_id                       
1082               83.031506  

         events_count          mean_interval
user_id                                     
1040               60 1 days 09:24:31.271186
1091               51 1 days 14:05:49.100000
1072               56 1 days 14:08:07.254545
1075               52 1 days 15:42:39.019607
1076               53 1 days 16:33:15.365384
         events_count          mean_interval
user_

In [4]:
# Связь среднего интервала с общим числом событий пользователя
corr = user_intervals["events_count"].corr(user_intervals["mean_interval_hours"])
print(f"Корреляция числа событий и среднего интервала: {corr:.2f}")

# Сколько интервалов укладывается в одну сессию (<= 30 минут)
share_short = (df_events_user["delta_since_prev"] <= pd.Timedelta(minutes=30)).mean()
print(f"Доля интервалов не длиннее 30 минут: {share_short:.1%}")

Корреляция числа событий и среднего интервала: -0.94
Доля интервалов не длиннее 30 минут: 1.1%


### Задание 3. Интерпретировать, что это может означать с точки зрения активности

- **Пользователь 1040**: самый короткий средний интервал, ≈ 33 часа (1 день 9 часов), и больше всех событий (60). Это самый активный и регулярный пользователь, он заходит на сайт почти каждый день. Такого клиента стоит удерживать: программа лояльности, персональные рекомендации.
- **Пользователь 1082**: самый длинный средний интервал, ≈ 83 часа (почти 3,5 дня), и всего 27 событий. Он заходит редко и от случая к случаю. Это «спящий» или случайный пользователь, которого магазин может потерять. Ему подойдут рассылки для возврата и напоминания о брошенной корзине.
- У всех пользователей средний интервал лежит в диапазоне от 33 до 83 часов, медиана ≈ 52 часа. То есть обычный пользователь возвращается примерно раз в 2 дня.
- Корреляция между числом событий и средним интервалом ≈ −0,94: чем чаще пользователь возвращается, тем больше событий он совершает. Поэтому средний интервал можно использовать как метрику вовлечённости (по смыслу это Recency/Frequency из RFM-анализа).
- Только ≈ 1% интервалов короче 30 минут, значит почти каждое событие открывает новую сессию. Средний интервал здесь показывает, как часто пользователь **заходит на сайт**, а не как быстро он действует внутри одного визита. Одна долгая пауза сильно увеличивает среднее, поэтому на реальных логах стоит смотреть ещё и медиану.